In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, struct, to_json
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

spark = SparkSession.builder.appName("FraudDetection").config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

users_df = spark.read.csv("data/user_table.csv", header=True, inferSchema=True)

tx_schema = StructType([
 StructField("tx_id", IntegerType(), True),
 StructField("userId", IntegerType(), True),
 StructField("amount", DoubleType(), True),
 StructField("timestamp", DoubleType(), True)
])

kafka_stream = spark.readStream.format("kafka").option("kafka.bootstrap.servers", "kafka:9092").option("subscribe", "fraud-detection") .load()

parsed_stream = kafka_stream.select(from_json(col("value").cast("string"),
tx_schema).alias("tx")).select("tx.*")
high_amount_stream = parsed_stream.filter(col("amount") > 10000)
high_amount_enriched = high_amount_stream.join(users_df, "userId")

user_alert_stream = parsed_stream.filter(
    (col("userId") == 103) &
    (col("amount") > 5000)
)
user_alert_enriched = user_alert_stream.join(
    users_df,
    "userId"
)
high_amount_output  = high_amount_enriched.withColumn("value", to_json(struct("*")).cast("string")).select("value")
user_alert_output = user_alert_enriched.withColumn(
    "value",
    to_json(struct("*")).cast("string")
).select("value")

query1 = high_amount_output.writeStream.format("kafka").option("kafka.bootstrap.servers", "kafka:9092").option("topic", "fraudDetection").option("checkpointLocation", "/workspace/checkpoints/high_amount").start()

query2 = user_alert_output.writeStream.format("kafka").option("kafka.bootstrap.servers","kafka:9092").option("topic","user-alert-topic").option("checkpointLocation","/workspace/checkpoints/user_alert") .start()

26/06/19 18:16:45 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/19 18:16:45 WARN StreamingQueryManager: Stopping existing streaming query [id=5fd5cb95-efca-478f-8f71-446f708b6344, runId=2c479def-ab0e-4302-b799-5d5c1e323f4e], as a new run is being started.
26/06/19 18:16:45 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/19 18:16:45 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/06/19 18:16:45 WARN StreamingQueryManager: Stopping existing streaming query [id=a464d77e-0464-4a1a-bd6c-edc062f5584e, runId=455d4a1e-7203-4dea-86f6-a26a39382d49], as a new run is being started.
26/06/19 18:16:45 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.comm